In [ ]:
!git clone https://github.com/YPolina/Medicine.git
%cd ./Medicine/BELKA/training
!pip install -r ../requirements.txt
from google.colab import drive
drive.mount('/content/drive')

In [9]:
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from lightgbm import LGBMClassifier
import pickle 
import os
from tqdm import tqdm
from fastparquet import write
import pyarrow as pa
import pyarrow.parquet as pq

In [12]:
fpg = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024, includeChirality=True)

def compute_fps(smiles_batch):

    """
    Compute fingerprints for a batch of SMILES
    """
    results = []
    for smiles in smiles_batch:
        mol = Chem.MolFromSmiles(smiles.replace('[Dy]', '[H]'))
        if mol is None:
            results.append((np.zeros(1024, dtype=np.int8), {}))
        else:
            fp = fpg.GetCountFingerprint(mol)
            arr = np.zeros(1024, dtype=np.int8)
            Chem.DataStructs.ConvertToNumpyArray(fp, arr)
            results.append(arr)
    return np.array(results)

def train_model_for_protein(protein_data, protein_name, batch_size=1000, lgb_params=None):
    """
    Train a LightGBM model for a protein using batched data

    Args:
        protein_data (pd.DataFrame): Data containing SMILES strings and labels
        protein_name (str): Name of the protein
        batch_size (int): Number of samples per batch
        lgb_params (dict): Parameters for the LightGBM classifier

    Returns:
        LGBMClassifier: Trained LightGBM model with the best iteration
    """
    #Split data into batches
    smiles_batches = np.array_split(protein_data['molecule_smiles'].tolist(), -(-len(protein_data) // batch_size))
    label_batches = np.array_split(protein_data['binds'].tolist(), -(-len(protein_data) // batch_size))

    #Initialize the LightGBM
    lgb_cls = LGBMClassifier(**lgb_params)

    #Train the model in batches
    for smiles_batch, label_batch in tqdm(zip(smiles_batches, label_batches), total=len(smiles_batches), desc=f"Training {protein_name}"):
        if len(smiles_batch) != len(label_batch):
            continue

        #Compute fingerprints for the batch
        X_batch = compute_fps(smiles_batch)
        y_batch = np.array(label_batch)

        num_features = X_batch.shape[1]
        df_X_batch = pd.DataFrame(X_batch, columns=[f"feature_{j}" for j in range(num_features)])
        df_y_batch = pd.DataFrame(y_batch, columns=["label"])

        # Train LightGBM
        lgb_cls.fit(
            df_X_batch, 
            df_y_batch.values.ravel(), 
            eval_metric='auc', 
            init_model=lgb_cls.booster_ if hasattr(lgb_cls, "booster_") else None
        )


    best_iteration = lgb_cls.best_iteration_
    print(f"Best iteration: {best_iteration}")

    lgb_cls.set_params(n_estimators=best_iteration)

    return lgb_cls

def save_models(model, protein, save_dir="../checkpoints"):
    os.makedirs(save_dir, exist_ok=True)

    model_path = os.path.join(save_dir, f"{protein}_lightgbm.txt")
    
    model.booster_.save_model(model_path)
    
    print(f"LightGBM model for {protein} saved at {model_path}")

def train_models_by_protein(batch_size=100000, save_dir="../checkpoints", lgb_params = None, eval_set = None):
    protein_names = ['sEH', 'BRD4', 'HSA']

    for protein in tqdm(protein_names, desc="Training models"):

        train_data = pd.read_parquet(f'../intermediates/train_data/{protein}/{protein}_train.parquet')
        val_data = pd.read_parquet(f'../intermediates/train_data/{protein}/{protein}_val.parquet')
        save_eval_set(val_data, protein, batch_size = 500)
        model = train_model_for_protein(train_data, protein, batch_size, lgb_params)

        save_models(model, protein, save_dir)

    return "Training complete"


In [5]:
def save_eval_set(eval_set, protein_name, batch_size=1000, save_path="../intermediates/embeddings/"):
    """Efficiently saves evaluation data in batches"""
    
    if eval_set is None or eval_set.empty:
        return None
    
    save_path = os.path.join(save_path, f"{protein_name}_lightgbm_val.parquet")
    
    eval_features = compute_fps(eval_set['molecule_smiles'])  
    eval_labels = eval_set['binds'].values 

   
    num_features = eval_features.shape[1]  
    col_names = [f"feature_{i}" for i in range(num_features)] + ["label"]

    for i in range(0, len(eval_features), batch_size):
        batch_features = eval_features[i : i + batch_size]
        batch_labels = eval_labels[i : i + batch_size]

        df_batch = pd.DataFrame(batch_features, columns=[f"feature_{j}" for j in range(num_features)])
        df_batch["label"] = batch_labels

        table = pa.Table.from_pandas(df_batch)
        
        if os.path.exists(save_path):
            with open(save_path, 'ab') as f:
                pq.write_table(table, f)
        else:
            with open(save_path, 'wb') as f:
                pq.write_table(table, f)

        del batch_features, batch_labels, df_batch, table 

    print(f"Evaluation set saved to {save_path}")

In [13]:
lgb_params = {
        'max_depth': 11,
        'bagging_fraction': 0.9,
        'learning_rate': 0.05,
        'colsample_bytree': 1,
        'colsample_bynode': 0.5,
        'lambda_l1': 1,
        'objective': 'binary',
        'lambda_l2': 1.5,
        'num_leaves': 490,
        'min_data_in_leaf': 50,
        'verbose': -1,
        'metric': 'average_precision',
        'device': 'cpu'
    }

train_models_by_protein(lgb_params=lgb_params)

Training models:   0%|          | 0/3 [00:00<?, ?it/s]

Evaluation set saved to ../intermediates/embeddings/sEH_lightgbm_val.parquet


Training models:  33%|███▎      | 1/3 [32:30<1:05:01, 1950.97s/it]

Best iteration: 0
LightGBM model for sEH saved at ../checkpoints/sEH_lightgbm.txt
Evaluation set saved to ../intermediates/embeddings/BRD4_lightgbm_val.parquet


Training BRD4: 100%|██████████| 35/35 [27:43<00:00, 47.51s/it]


...

Training models:  67%|██████▋   | 2/3 [1:03:19<31:30, 1890.81s/it]

LightGBM model for BRD4 saved at ../checkpoints/BRD4_lightgbm.txt
Evaluation set saved to ../intermediates/embeddings/HSA_lightgbm_val.parquet


Training models: 100%|██████████| 3/3 [1:34:06<00:00, 1882.18s/it]


Best iteration: 0
LightGBM model for HSA saved at ../checkpoints/HSA_lightgbm.txt


'Training complete'